# Exercise 1 — Text Preprocessing

Raw Text → Lowercase → Punctuation Handling → Tokens → Stopword Choice → **Stems OR Lemmas** → Vocabulary → Frequencies.

## Learning objectives
- Inspect each transformation before numerical representation.
- Explain task-dependent stopword choices and morphology assumptions.
- Build a deterministic vocabulary and inspect repeated-word counts.
- Compare changes on the same inputs using functions from `src.preprocessing`.

This is the complete working/reference notebook. Student cores 1.1–1.7 are identified in `src/preprocessing.py`; a dedicated starter checkpoint comes later. No algorithms are independently implemented here. Stemming and lemmatization are parallel alternatives, not sequential cleaning steps.

## Setup supplied by the starter

From the repository root activate the environment, install requirements, then run:

```sh
python -m src.nltk_resources --download
python -m src.nltk_resources
```

Only English stopwords and WordNet are required. No downloads occur in notebook cells. Open Jupyter from the repository root or `notebooks/`; the next cell resolves imports from either location. Jupyter is optional tooling, separate from application requirements; see the setup checklist.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open the notebook from the repository root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nltk_resources import NLTKResourceError, require_resource
from src.preprocessing import (
    lowercase_text, remove_punctuation, tokenize_words, remove_stopwords,
    stem_words, lemmatize_words, build_vocabulary, calculate_word_frequencies,
    tag_parts_of_speech,
)
require_resource('stopwords')
require_resource('wordnet')

## 1. Raw text

Load an existing workshop input and keep the original so every stage can be compared with it.

In [ ]:
examples = json.loads((ROOT / 'data/examples/preprocessing_examples.json').read_text(encoding='utf-8'))['examples']
raw_text = examples[1]['text']
raw_text

## 2. Lowercase normalization — boundary 1.1

Reduce separate forms such as Students and students. Case can matter for names or acronyms. This call changes case only.

In [ ]:
lowercase = lowercase_text(raw_text)
lowercase

## 3. Punctuation handling — boundary 1.2

Replace punctuation with spaces instead of merging neighbours. Whitespace remains visible. Apostrophes, decimals and hyphens can carry meaning; inspect the policy's limitations.

In [ ]:
punctuation_handled = remove_punctuation(lowercase)
punctuation_handled

## 4. Word tokenization — boundary 1.3

NLTK WordPunct returns ordered tokens with repetitions. No sentence-tokenizer model is needed. Retained punctuation can also become a token.

In [ ]:
tokens = tokenize_words(punctuation_handled)
tokens

## 5. Stopword choice — boundary 1.4

The starter supplies English stopwords. Filtering is task-dependent: removing not can change meaning. The input list is not modified.

In [ ]:
without_stopwords = remove_stopwords(tokens)
without_stopwords

## 6. Stemming — boundary 1.5

Porter stemming applies rules, and stems need not be dictionary words. Its default behaviour also lowercases tokens. Preserve the original tokens for comparison.

In [ ]:
stemmed_tokens = stem_words(without_stopwords)
stemmed_tokens

## 7. Lemmatization — boundary 1.6

Apply WordNet to the same tokens, not to their stems. Noun is the default assumption. Choosing one POS for all tokens is a context demonstration, not grammatical analysis.

In [ ]:
lemmatized_tokens = lemmatize_words(without_stopwords, pos='n')
lemmatized_tokens

## 8. Vocabulary — boundary 1.7

Vocabulary is the unique token set of the chosen representation; here we choose lemmas. Sorting supplies stable column positions for later numerical representations.

In [ ]:
selected_tokens = lemmatized_tokens
vocabulary = build_vocabulary(selected_tokens)
vocabulary

## 9. Word frequency — boundary 1.7

Every repeated token contributes another occurrence. Compare the total number of tokens with the number of vocabulary items.

In [ ]:
frequencies = calculate_word_frequencies(selected_tokens)
assert sum(frequencies.values()) == len(selected_tokens)
frequencies

## 10. Compare stems and lemmas

Each row starts from one original token. Neither operation is universally better. Noun lemmatization may leave processing unchanged; a verb assumption can yield process.

In [ ]:
comparison = [
    {'Original': token, 'Stem': stem, 'Lemma (n)': lemma}
    for token, stem, lemma in zip(without_stopwords, stemmed_tokens, lemmatized_tokens)
]
comparison

In [ ]:
probe = ['studies', 'processing', 'learning']
{
    'tokens': probe,
    'stems': stem_words(probe),
    'noun lemmas': lemmatize_words(probe, pos='n'),
    'verb lemmas': lemmatize_words(probe, pos='v'),
}

## 11. Small experiments

Change capitalization, add punctuation and repeat words. First implement the small counting loop in the source boundary, then compare with Counter as a library check. Keep stopword filtering optional and inspect negation.

In [ ]:
from collections import Counter
assert frequencies == dict(Counter(selected_tokens))

capitalized_tokens = tokenize_words('NLP nlp NLP')
{
    'original vocabulary': build_vocabulary(capitalized_tokens),
    'normalized vocabulary': build_vocabulary(tokenize_words(lowercase_text('NLP nlp NLP'))),
}

In [ ]:
negation_tokens = tokenize_words('not useful')
{
    'original': negation_tokens,
    'english stopword filtering': remove_stopwords(negation_tokens),
    'empty stopword set': remove_stopwords(negation_tokens, set()),
}

In [ ]:
punctuation_probe = 'human-language! NLP...'
{
    'raw tokens': tokenize_words(punctuation_probe),
    'punctuation-handled text': remove_punctuation(punctuation_probe),
    'handled tokens': tokenize_words(remove_punctuation(punctuation_probe)),
}

### Vocabulary across documents

The function accepts a flat token sequence, not raw document strings. Assemble the inputs below, then reuse the same source function.

In [ ]:
documents = ['NLP processes language', 'NLP understands text']
corpus_tokens = [token for document in documents for token in tokenize_words(lowercase_text(document))]
build_vocabulary(corpus_tokens)

## Optional instructor POS demo

Install the small English tagger explicitly with `python -m src.nltk_resources --download --include-pos`. If absent, this cell prints guidance and the core notebook still works. Predictions on invented names can be imperfect. This is POS, not NER; richer NER is deferred to the Transformer task.

In [ ]:
demo_sentence = 'Dr. Nivora Pellin teaches Natural Language Processing at Hindu College of Engineering.'
try:
    print(tag_parts_of_speech(tokenize_words(demo_sentence)))
except NLTKResourceError as error:
    print(error)

## Checkpoint 1 — Text preprocessing works

From the repository root run:

```sh
python -m pytest tests/test_preprocessing.py
python -m streamlit run app.py
```

Open the Preprocessing Lab and inspect every stage. Reference implementations remain complete. The next exercise will use these tokens and vocabulary for numerical representations; it is not implemented here.